# Tarea 1 – Parte 1: Recolección y Limpieza

**Medio elegido:** El Comercio  
**Fecha de descarga:** 2026-10-17

**Categorías finales:**
- politica
- economia
- deportes
- ciencia_tecnologia

**Fuente principal:** RSS oficial de El Comercio.

Se utilizarán los canales de Política, Economía, DT y Tecnología y Ciencias.

In [42]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import time
import re
from datetime import datetime

In [43]:
BASE = Path(".")
DATOS = BASE / "datos"
RAW = DATOS / "raw"

DATOS.mkdir(exist_ok=True)
RAW.mkdir(exist_ok=True)

print("Carpeta datos:", DATOS)
print("Carpeta raw:", RAW)

Carpeta datos: datos
Carpeta raw: datos\raw


In [44]:
CATEGORIAS = {
    "politica": "politica",
    "economia": "economia",
    "dt": "deportes",
    "tecnologia": "ciencia_tecnologia"
}

CATEGORIAS

{'politica': 'politica',
 'economia': 'economia',
 'dt': 'deportes',
 'tecnologia': 'ciencia_tecnologia'}

## Recolección mediante RSS

Se utilizarán los canales RSS oficiales de El Comercio para obtener noticias de las secciones:
- Política
- Economía
- DT
- Tecnología y Ciencias

In [45]:
RSS_URL = "https://elcomercio.pe/rss/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; TareaAcademicaNoticias/1.0)"
}

respuesta = requests.get(
    RSS_URL,
    headers=HEADERS,
    timeout=20
)

print("Status:", respuesta.status_code)
print("Tamaño:", len(respuesta.text))

Status: 200
Tamaño: 1779113


In [46]:
soup = BeautifulSoup(respuesta.text, "html.parser")

links = []

for a in soup.find_all("a", href=True):
    texto = a.get_text(" ", strip=True)
    href = a["href"]

    if texto:
        links.append((texto, href))

links[:20]

[('Mis beneficios Club',
  '/suscripciones/?utm_source=boton-home&utm_medium=experiencias-piano-boton-home-generico&utm_campaign=ec-cyber-news-club'),
 ('Iniciar sesión', '/login'),
 ('Para ti', '/para-ti/'),
 ('Edición impresa', 'https://web.peruquiosco.pe/'),
 ('Video', '/videos'),
 ('Colecciones El Comercio', 'https://coleccioneselcomercio.pe/'),
 ('Newsletters', '/newsletters'),
 ('Juegos', '/juegos'),
 ('Club El Comercio', 'https://clubelcomercio.pe/'),
 ('Portada', '/'),
 ('Lo último', '/ultimas-noticias/'),
 ('Editorial', '/opinion/editorial/'),
 ('Opinión', '/opinion/'),
 ('Columnistas', '/opinion/columnistas/'),
 ('Voz Universitaria', '/opinion/voz-universitaria/'),
 ('Vuelo de libelo', '/opinion/vuelo-de-libelo/'),
 ('Efemérides', '/opinion/efemerides/'),
 ('El informante', '/opinion/el-informante/'),
 ('Piedra de toque', '/opinion/piedra-de-toque/'),
 ('Política', '/politica/')]

In [47]:
objetivos = ["politica", "economia", "dt", "tecnologia"]

links_objetivo = []

for texto, href in links:
    texto_limpio = texto.lower()
    href_limpio = href.lower()

    if any(palabra in texto_limpio or palabra in href_limpio for palabra in objetivos):
        links_objetivo.append((texto, href))

links_objetivo[:50]

[('Política', '/politica/'),
 ('Gobierno', '/politica/gobierno/'),
 ('Congreso', '/politica/congreso/'),
 ('Justicia', '/politica/justicia/'),
 ('Elecciones', '/politica/elecciones/'),
 ('Impuestos', '/politica/impuestos/'),
 ('Polémica', '/politica/polemica/'),
 ('Partidos', '/politica/partidos/'),
 ('Internacional', '/politica/internacional/'),
 ('Actualidad', '/politica/actualidad/'),
 ('Opinión', '/politica/opinion/'),
 ('Economía', '/economia/'),
 ('Mundo', '/economia/mundo/'),
 ('Perú', '/economia/peru/'),
 ('Mercados', '/economia/mercados/'),
 ('Finanzas Personales', '/economia/personal/'),
 ('Ejecutivos', '/economia/ejecutivos/'),
 ('Negocios', '/economia/negocios/'),
 ('Opinión', '/economia/opinion/'),
 ('LEC', '/economia/lec/'),
 ('Día 1', '/economia/dia-1/'),
 ('DT', '/deporte-total/'),
 ('DT Running', '/deporte-total/dt-running/'),
 ('Tecnología', '/respuestas/tecnologia/'),
 ('Tecnología y ciencias', '/tecnologia/'),
 ('Streamers', '/tecnologia/streamers/'),
 ('Inteligenci

Filtré los enlaces encontrados para identificar únicamente aquellos relacionados con las secciones Política, Economía, DT y Tecnología, que corresponden a las cuatro categorías finales del trabajo.

In [48]:
for texto, href in links_objetivo:
    print(texto, "->", href)

Política -> /politica/
Gobierno -> /politica/gobierno/
Congreso -> /politica/congreso/
Justicia -> /politica/justicia/
Elecciones -> /politica/elecciones/
Impuestos -> /politica/impuestos/
Polémica -> /politica/polemica/
Partidos -> /politica/partidos/
Internacional -> /politica/internacional/
Actualidad -> /politica/actualidad/
Opinión -> /politica/opinion/
Economía -> /economia/
Mundo -> /economia/mundo/
Perú -> /economia/peru/
Mercados -> /economia/mercados/
Finanzas Personales -> /economia/personal/
Ejecutivos -> /economia/ejecutivos/
Negocios -> /economia/negocios/
Opinión -> /economia/opinion/
LEC -> /economia/lec/
Día 1 -> /economia/dia-1/
DT -> /deporte-total/
DT Running -> /deporte-total/dt-running/
Tecnología -> /respuestas/tecnologia/
Tecnología y ciencias -> /tecnologia/
Streamers -> /tecnologia/streamers/
Inteligencia Artificial -> /tecnologia/inteligencia-artificial/
Ciberseguridad -> /tecnologia/ciberseguridad/
Gadgets -> /tecnologia/gadgets/
Videojuegos -> /tecnologia/v

In [49]:
RSS_FEEDS = {
    "politica": "https://elcomercio.pe/arc/outboundfeeds/rss/category/politica/?outputType=xml",
    "economia": "https://elcomercio.pe/arc/outboundfeeds/rss/category/economia/?outputType=xml",
    "deportes": "https://elcomercio.pe/arc/outboundfeeds/rss/category/deporte-total/?outputType=xml",
    "ciencia_tecnologia": "https://elcomercio.pe/arc/outboundfeeds/rss/category/tecnologia/?outputType=xml"
}

RSS_FEEDS

{'politica': 'https://elcomercio.pe/arc/outboundfeeds/rss/category/politica/?outputType=xml',
 'economia': 'https://elcomercio.pe/arc/outboundfeeds/rss/category/economia/?outputType=xml',
 'deportes': 'https://elcomercio.pe/arc/outboundfeeds/rss/category/deporte-total/?outputType=xml',
 'ciencia_tecnologia': 'https://elcomercio.pe/arc/outboundfeeds/rss/category/tecnologia/?outputType=xml'}

In [50]:
for categoria, url in RSS_FEEDS.items():
    r = requests.get(url, headers=HEADERS, timeout=20)
    print(categoria, "->", r.status_code, "| tamaño:", len(r.text))

politica -> 200 | tamaño: 213080
economia -> 200 | tamaño: 423206
deportes -> 200 | tamaño: 537364
ciencia_tecnologia -> 200 | tamaño: 118578


In [51]:
for categoria, url in RSS_FEEDS.items():
    r = requests.get(url, headers=HEADERS, timeout=20)
    soup_rss = BeautifulSoup(r.content, "xml")
    items = soup_rss.find_all("item")
    print(categoria, "->", len(items), "noticias")

politica -> 24 noticias
economia -> 55 noticias
deportes -> 72 noticias
ciencia_tecnologia -> 17 noticias


## Extracción inicial de noticias desde RSS

Los RSS oficiales disponibles contienen 168 noticias. Debido a que la tarea requiere
como mínimo 350 registros, estas noticias se utilizarán como primera fuente y luego
se complementarán con publicaciones adicionales de las secciones de El Comercio.

In [52]:
registros = []

SECCION_ORIGINAL = {
    "politica": "Política",
    "economia": "Economía",
    "deportes": "DT",
    "ciencia_tecnologia": "Tecnología y Ciencias"
}

for categoria, url in RSS_FEEDS.items():

    print(f"Descargando: {categoria}")

    r = requests.get(
        url,
        headers=HEADERS,
        timeout=20
    )

    # Guardar respaldo RAW
    archivo_raw = RAW / f"{categoria}.xml"

    with open(archivo_raw, "wb") as f:
        f.write(r.content)

    soup_rss = BeautifulSoup(r.content, "xml")
    items = soup_rss.find_all("item")

    for item in items:

        titulo = item.title.get_text(strip=True) if item.title else ""

        link = item.link.get_text(strip=True) if item.link else ""

        fecha = (
            item.pubDate.get_text(strip=True)
            if item.pubDate
            else ""
        )

        # Descripción / bajada
        if item.description:
            descripcion_html = item.description.get_text()
            bajada = BeautifulSoup(
                descripcion_html,
                "html.parser"
            ).get_text(" ", strip=True)
        else:
            bajada = ""

        registros.append({
            "titular": titulo,
            "bajada": bajada,
            "medio": "El Comercio",
            "seccion_original": SECCION_ORIGINAL[categoria],
            "url": link,
            "fecha": fecha,
            "etiqueta": categoria
        })

    # Pausa entre solicitudes
    time.sleep(1.5)

print("Registros obtenidos:", len(registros))

Descargando: politica
Descargando: economia
Descargando: deportes
Descargando: ciencia_tecnologia
Registros obtenidos: 168


In [53]:
df = pd.DataFrame(registros)

print("Dimensión:", df.shape)

df.head()

Dimensión: (168, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Facultades legislativas: Senado sigue sin alca...,El pedido de facultades legislativas del Ejecu...,El Comercio,Política,https://elcomercio.pe/politica/facultades-legi...,"Sat, 17 Oct 2026 19:19:52 +0000",politica
1,Renovación Popular se declara en reorganizació...,La medida deja sin efecto las credenciales de ...,El Comercio,Política,https://elcomercio.pe/politica/renovacion-popu...,"Sat, 17 Oct 2026 19:17:01 +0000",politica
2,"Alfonso, ¿ahora qué?, una crónica de Fernando ...",El líder de Ahora Nación y portavoz de su banc...,El Comercio,Política,https://elcomercio.pe/politica/alfonso-ahora-q...,"Sat, 17 Oct 2026 19:14:15 +0000",politica
3,Tomás Gálvez dice que Ministerio Público requi...,"El fiscal de la Nación, Tomás Gálvez, pidió al...",El Comercio,Política,https://elcomercio.pe/politica/actualidad/toma...,"Sat, 17 Oct 2026 18:51:28 +0000",politica
4,De Gregorio Santos a “El lunarejo”: más de 8 m...,Entre los solicitantes se encuentra Gregorio S...,El Comercio,Política,https://elcomercio.pe/politica/indultos-presid...,"Sat, 17 Oct 2026 17:27:55 +0000",politica


In [54]:
df["etiqueta"].value_counts()

etiqueta
deportes              72
economia              55
politica              24
ciencia_tecnologia    17
Name: count, dtype: int64

## Recolección complementaria desde páginas de sección

El RSS no contiene suficientes registros para alcanzar el mínimo requerido de 350 noticias.
Por ello, se complementará la base con noticias publicadas en las páginas de sección de
El Comercio, respetando pausas entre solicitudes y eliminando posteriormente los duplicados.

In [55]:
SECCIONES = {
    "politica": "https://elcomercio.pe/politica/",
    "economia": "https://elcomercio.pe/economia/",
    "deportes": "https://elcomercio.pe/deporte-total/",
    "ciencia_tecnologia": "https://elcomercio.pe/tecnologia/"
}

SECCIONES

{'politica': 'https://elcomercio.pe/politica/',
 'economia': 'https://elcomercio.pe/economia/',
 'deportes': 'https://elcomercio.pe/deporte-total/',
 'ciencia_tecnologia': 'https://elcomercio.pe/tecnologia/'}

In [56]:
url_prueba = SECCIONES["politica"]

r = requests.get(
    url_prueba,
    headers=HEADERS,
    timeout=20
)

print("Status:", r.status_code)
print("Tamaño:", len(r.text))

Status: 200
Tamaño: 579617


In [57]:
soup = BeautifulSoup(r.text, "html.parser")

enlaces_politica = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    texto = a.get_text(" ", strip=True)

    if (
        texto
        and "/politica/" in href
        and href != "/politica/"
    ):
        if href.startswith("/"):
            href = "https://elcomercio.pe" + href

        enlaces_politica.append((texto, href))

len(enlaces_politica)

153

In [58]:
enlaces_politica = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    texto = a.get_text(" ", strip=True)

    if not texto:
        continue

    if href.startswith("/"):
        href = "https://elcomercio.pe" + href

    # Nos quedamos solo con enlaces de noticias reales
    if (
        "elcomercio.pe/politica/" in href
        and "-noticia/" in href
    ):
        enlaces_politica.append((texto, href))

# Eliminar URLs repetidas
enlaces_politica = list(dict.fromkeys(enlaces_politica))

print("Enlaces encontrados:", len(enlaces_politica))

enlaces_politica[:20]

Enlaces encontrados: 57


[('Vargas Llosa',
  'https://elcomercio.pe/politica/alvaro-vargas-llosa-es-nombrado-embajador-del-peru-en-eeuu-cuales-seran-las-prioridades-de-su-agenda-noticia/'),
 ('Facultades legislativas: Senado sigue sin alcanzar consenso y debate continúa entrampado',
  'https://elcomercio.pe/politica/facultades-legislativas-senado-sigue-sin-alcanzar-consenso-y-debate-continua-entrampado-apuesplan-noticia/'),
 ('Renovación Popular se declara en reorganización partidaria: ¿A qué se debe y cuáles son los alcances de la decisión?',
  'https://elcomercio.pe/politica/renovacion-popular-se-declara-en-reorganizacion-partidaria-a-que-se-debe-y-cuales-son-los-alcances-de-la-decision-noticia/'),
 ('Suscriptores',
  'https://elcomercio.pe/politica/alfonso-ahora-que-una-cronica-de-fernando-vivas-sobre-lopez-chau-lider-de-ahora-nacion-apuesplan-noticia/'),
 ('Alfonso, ¿ahora qué?, una crónica de Fernando Vivas sobre López-Chau, líder de Ahora Nación',
  'https://elcomercio.pe/politica/alfonso-ahora-que-una-c

In [59]:
mejor_texto_por_url = {}

for texto, url in enlaces_politica:

    if (
        url not in mejor_texto_por_url
        or len(texto) > len(mejor_texto_por_url[url])
    ):
        mejor_texto_por_url[url] = texto

enlaces_politica_limpios = [
    (texto, url)
    for url, texto in mejor_texto_por_url.items()
]

print("Noticias únicas:", len(enlaces_politica_limpios))

enlaces_politica_limpios[:10]

Noticias únicas: 50


[('Álvaro Vargas Llosa es nombrado embajador del Perú en EE.UU.: ¿Cuáles serán las prioridades de su agenda?',
  'https://elcomercio.pe/politica/alvaro-vargas-llosa-es-nombrado-embajador-del-peru-en-eeuu-cuales-seran-las-prioridades-de-su-agenda-noticia/'),
 ('Facultades legislativas: Senado sigue sin alcanzar consenso y debate continúa entrampado',
  'https://elcomercio.pe/politica/facultades-legislativas-senado-sigue-sin-alcanzar-consenso-y-debate-continua-entrampado-apuesplan-noticia/'),
 ('Renovación Popular se declara en reorganización partidaria: ¿A qué se debe y cuáles son los alcances de la decisión?',
  'https://elcomercio.pe/politica/renovacion-popular-se-declara-en-reorganizacion-partidaria-a-que-se-debe-y-cuales-son-los-alcances-de-la-decision-noticia/'),
 ('Alfonso, ¿ahora qué?, una crónica de Fernando Vivas sobre López-Chau, líder de Ahora Nación',
  'https://elcomercio.pe/politica/alfonso-ahora-que-una-cronica-de-fernando-vivas-sobre-lopez-chau-lider-de-ahora-nacion-apue

In [60]:
for titular, url in enlaces_politica_limpios[:10]:
    print(titular)
    print(url)
    print("-" * 80)

Álvaro Vargas Llosa es nombrado embajador del Perú en EE.UU.: ¿Cuáles serán las prioridades de su agenda?
https://elcomercio.pe/politica/alvaro-vargas-llosa-es-nombrado-embajador-del-peru-en-eeuu-cuales-seran-las-prioridades-de-su-agenda-noticia/
--------------------------------------------------------------------------------
Facultades legislativas: Senado sigue sin alcanzar consenso y debate continúa entrampado
https://elcomercio.pe/politica/facultades-legislativas-senado-sigue-sin-alcanzar-consenso-y-debate-continua-entrampado-apuesplan-noticia/
--------------------------------------------------------------------------------
Renovación Popular se declara en reorganización partidaria: ¿A qué se debe y cuáles son los alcances de la decisión?
https://elcomercio.pe/politica/renovacion-popular-se-declara-en-reorganizacion-partidaria-a-que-se-debe-y-cuales-son-los-alcances-de-la-decision-noticia/
--------------------------------------------------------------------------------
Alfonso, ¿ah

In [61]:
def obtener_enlaces_seccion(categoria, url):
    r = requests.get(
        url,
        headers=HEADERS,
        timeout=20
    )

    print(categoria, "->", r.status_code)

    soup = BeautifulSoup(r.text, "html.parser")

    mejor_texto_por_url = {}

    for a in soup.find_all("a", href=True):
        href = a["href"]
        texto = a.get_text(" ", strip=True)

        if not texto:
            continue

        if href.startswith("/"):
            href = "https://elcomercio.pe" + href

        # Solo artículos/noticias
        if (
            "elcomercio.pe/" in href
            and "-noticia/" in href
        ):
            if (
                href not in mejor_texto_por_url
                or len(texto) > len(mejor_texto_por_url[href])
            ):
                mejor_texto_por_url[href] = texto

    noticias = []

    for url_noticia, titular in mejor_texto_por_url.items():
        noticias.append({
            "titular": titular,
            "url": url_noticia,
            "etiqueta": categoria
        })

    time.sleep(1.5)

    return noticias

In [62]:
enlaces_secciones = []

for categoria, url in SECCIONES.items():
    noticias = obtener_enlaces_seccion(categoria, url)

    print(
        categoria,
        "->",
        len(noticias),
        "noticias encontradas"
    )

    enlaces_secciones.extend(noticias)

politica -> 200
politica -> 72 noticias encontradas
economia -> 200
economia -> 78 noticias encontradas
deportes -> 200
deportes -> 51 noticias encontradas
ciencia_tecnologia -> 200
ciencia_tecnologia -> 48 noticias encontradas


In [63]:
df_secciones = pd.DataFrame(enlaces_secciones)

print("Total encontrado:", len(df_secciones))

df_secciones.head()

Total encontrado: 249


,titular,url,etiqueta
0,ONPE,https://elcomercio.pe/elecciones/resultados-on...,politica
1,Papa León,https://elcomercio.pe/mundo/europa/papa-leon-x...,politica
2,Tabla Liga 1,https://elcomercio.pe/deporte-total/futbol-per...,politica
3,Iron Maiden,https://elcomercio.pe/luces/musica/iron-maiden...,politica
4,Dólar,https://elcomercio.pe/economia/mercados/precio...,politica


In [64]:
df_secciones["etiqueta"].value_counts()

etiqueta
economia              78
politica              72
deportes              51
ciencia_tecnologia    48
Name: count, dtype: int64

In [65]:
urls_rss = set(df["url"])

df_secciones["ya_estaba_rss"] = df_secciones["url"].isin(urls_rss)

print("Ya estaban en RSS:")
print(df_secciones["ya_estaba_rss"].value_counts())

print()

print(
    "Noticias realmente nuevas:",
    (~df_secciones["ya_estaba_rss"]).sum()
)

Ya estaban en RSS:
ya_estaba_rss
False    131
True     118
Name: count, dtype: int64

Noticias realmente nuevas: 131


In [66]:
df_nuevas = df_secciones[
    ~df_secciones["ya_estaba_rss"]
].copy()

print(df_nuevas["etiqueta"].value_counts())

etiqueta
politica              46
economia              37
ciencia_tecnologia    30
deportes              18
Name: count, dtype: int64


In [67]:
urls_rss = set(df["url"])

df_secciones["ya_estaba_rss"] = df_secciones["url"].isin(urls_rss)

nuevas = (~df_secciones["ya_estaba_rss"]).sum()

print("Noticias realmente nuevas:", nuevas)
print("Total estimado RSS + nuevas:", len(df) + nuevas)

Noticias realmente nuevas: 131
Total estimado RSS + nuevas: 299


In [68]:
url_politica_pagina2 = "https://elcomercio.pe/politica/?page=2"

r2 = requests.get(
    url_politica_pagina2,
    headers=HEADERS,
    timeout=20
)

print("Status:", r2.status_code)
print("Tamaño:", len(r2.text))

Status: 200
Tamaño: 579617


In [69]:
soup2 = BeautifulSoup(r2.text, "html.parser")

enlaces_pagina2 = {}

for a in soup2.find_all("a", href=True):
    href = a["href"]
    texto = a.get_text(" ", strip=True)

    if not texto:
        continue

    if href.startswith("/"):
        href = "https://elcomercio.pe" + href

    if (
        "elcomercio.pe/politica/" in href
        and "-noticia/" in href
    ):
        if (
            href not in enlaces_pagina2
            or len(texto) > len(enlaces_pagina2[href])
        ):
            enlaces_pagina2[href] = texto

print("Noticias encontradas página 2:", len(enlaces_pagina2))

Noticias encontradas página 2: 50


In [70]:
urls_politica_actuales = set(
    df_secciones.loc[
        df_secciones["etiqueta"] == "politica",
        "url"
    ]
)

urls_pagina2 = set(enlaces_pagina2.keys())

nuevas_pagina2 = urls_pagina2 - urls_politica_actuales

print("Noticias distintas en página 2:", len(nuevas_pagina2))

Noticias distintas en página 2: 0


In [71]:
SUBSECCIONES = {
    "politica": [
        "https://elcomercio.pe/politica/gobierno/",
        "https://elcomercio.pe/politica/congreso/",
        "https://elcomercio.pe/politica/justicia/",
        "https://elcomercio.pe/politica/elecciones/"
    ],

    "economia": [
        "https://elcomercio.pe/economia/peru/",
        "https://elcomercio.pe/economia/mercados/",
        "https://elcomercio.pe/economia/negocios/",
        "https://elcomercio.pe/economia/personal/"
    ],

    "ciencia_tecnologia": [
        "https://elcomercio.pe/tecnologia/actualidad/",
        "https://elcomercio.pe/tecnologia/ciencias/",
        "https://elcomercio.pe/tecnologia/inteligencia-artificial/",
        "https://elcomercio.pe/tecnologia/gadgets/"
    ]
}

In [72]:
def extraer_enlaces_subseccion(categoria, url):

    r = requests.get(
        url,
        headers=HEADERS,
        timeout=20
    )

    print(url, "->", r.status_code)

    soup = BeautifulSoup(r.text, "html.parser")

    mejores = {}

    for a in soup.find_all("a", href=True):

        href = a["href"]
        texto = a.get_text(" ", strip=True)

        if not texto:
            continue

        if href.startswith("/"):
            href = "https://elcomercio.pe" + href

        if (
            "elcomercio.pe/" in href
            and "-noticia/" in href
        ):
            if (
                href not in mejores
                or len(texto) > len(mejores[href])
            ):
                mejores[href] = texto

    noticias = []

    for url_noticia, titular in mejores.items():

        noticias.append({
            "titular": titular,
            "url": url_noticia,
            "etiqueta": categoria
        })

    time.sleep(1.5)

    return noticias

In [73]:
registros_subsecciones = []

for categoria, urls in SUBSECCIONES.items():

    for url in urls:

        noticias = extraer_enlaces_subseccion(
            categoria,
            url
        )

        registros_subsecciones.extend(noticias)

https://elcomercio.pe/politica/gobierno/ -> 200
https://elcomercio.pe/politica/congreso/ -> 200
https://elcomercio.pe/politica/justicia/ -> 200
https://elcomercio.pe/politica/elecciones/ -> 200
https://elcomercio.pe/economia/peru/ -> 200
https://elcomercio.pe/economia/mercados/ -> 200
https://elcomercio.pe/economia/negocios/ -> 200
https://elcomercio.pe/economia/personal/ -> 200
https://elcomercio.pe/tecnologia/actualidad/ -> 200
https://elcomercio.pe/tecnologia/ciencias/ -> 200
https://elcomercio.pe/tecnologia/inteligencia-artificial/ -> 200
https://elcomercio.pe/tecnologia/gadgets/ -> 200


In [74]:
df_sub = pd.DataFrame(registros_subsecciones)

print("Total encontrados:", len(df_sub))

df_sub.head()

Total encontrados: 777


,titular,url,etiqueta
0,ONPE,https://elcomercio.pe/elecciones/resultados-on...,politica
1,Papa León,https://elcomercio.pe/mundo/europa/papa-leon-x...,politica
2,Tabla Liga 1,https://elcomercio.pe/deporte-total/futbol-per...,politica
3,Iron Maiden,https://elcomercio.pe/luces/musica/iron-maiden...,politica
4,Dólar,https://elcomercio.pe/economia/mercados/precio...,politica


In [75]:
urls_existentes = set(df["url"]) | set(df_secciones["url"])

df_sub["ya_existia"] = df_sub["url"].isin(urls_existentes)

df_sub_nuevas = df_sub[
    ~df_sub["ya_existia"]
].copy()

df_sub_nuevas = df_sub_nuevas.drop_duplicates(
    subset="url"
)

print(
    "Noticias nuevas de subsecciones:",
    len(df_sub_nuevas)
)

print()

print(
    df_sub_nuevas["etiqueta"].value_counts()
)

Noticias nuevas de subsecciones: 408

etiqueta
ciencia_tecnologia    149
economia              132
politica              127
Name: count, dtype: int64


In [76]:
total_estimado = (
    len(df)
    + len(df_nuevas)
    + len(df_sub_nuevas)
)

print("Total estimado:", total_estimado)

Total estimado: 707


In [77]:
# Base del RSS
base_rss = df[
    ["titular", "url", "etiqueta"]
].copy()

# Noticias nuevas de las páginas principales
base_secciones = df_nuevas[
    ["titular", "url", "etiqueta"]
].copy()

# Noticias nuevas obtenidas desde subsecciones
base_subsecciones = df_sub_nuevas[
    ["titular", "url", "etiqueta"]
].copy()

# Unir las tres fuentes
df_total = pd.concat(
    [
        base_rss,
        base_secciones,
        base_subsecciones
    ],
    ignore_index=True
)

# Eliminar URLs duplicadas
df_total = df_total.drop_duplicates(
    subset="url",
    keep="first"
).reset_index(drop=True)

print("Total único:", len(df_total))
print()
print(df_total["etiqueta"].value_counts())

Total único: 658

etiqueta
economia              208
politica              197
ciencia_tecnologia    179
deportes               74
Name: count, dtype: int64


In [78]:
muestra_politica = (
    df_total[df_total["etiqueta"] == "politica"]
    .sample(n=95, random_state=42)
)

muestra_economia = (
    df_total[df_total["etiqueta"] == "economia"]
    .sample(n=95, random_state=42)
)

muestra_tecnologia = (
    df_total[df_total["etiqueta"] == "ciencia_tecnologia"]
    .sample(n=95, random_state=42)
)

muestra_deportes = (
    df_total[df_total["etiqueta"] == "deportes"]
    .sample(n=75, random_state=42)
)

df_muestra = pd.concat(
    [
        muestra_politica,
        muestra_economia,
        muestra_tecnologia,
        muestra_deportes
    ],
    ignore_index=True
)

print("Total seleccionado:", len(df_muestra))
print()
print(df_muestra["etiqueta"].value_counts())

ValueError: Cannot take a larger sample than population when 'replace=False'

In [ ]:
print("URLs duplicadas:", df_muestra["url"].duplicated().sum())
print("Titulares vacíos:", df_muestra["titular"].isna().sum())

URLs duplicadas: 0
Titulares vacíos: 0


## Recuperación de metadatos de las noticias seleccionadas

Para completar la base final se consultan las páginas individuales de las noticias
seleccionadas. Se recuperan principalmente la bajada, fecha de publicación y sección
original mediante las etiquetas meta del artículo.

Se utiliza una pausa de un segundo entre solicitudes para evitar sobrecargar el sitio.

In [79]:
def obtener_meta_articulo(url, etiqueta):
    try:
        r = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        if r.status_code != 200:
            return {
                "bajada": "",
                "fecha": "",
                "seccion_original": ""
            }

        soup = BeautifulSoup(r.text, "html.parser")

        # ---------------------------
        # BAJADA / DESCRIPCIÓN
        # ---------------------------
        bajada = ""

        meta_desc = soup.find(
            "meta",
            attrs={"name": "description"}
        )

        if meta_desc and meta_desc.get("content"):
            bajada = meta_desc["content"].strip()

        if not bajada:
            meta_og = soup.find(
                "meta",
                attrs={"property": "og:description"}
            )

            if meta_og and meta_og.get("content"):
                bajada = meta_og["content"].strip()

        # ---------------------------
        # FECHA
        # ---------------------------
        fecha = ""

        posibles_fechas = [
            ("property", "article:published_time"),
            ("name", "date"),
            ("name", "pubdate"),
            ("name", "publish-date")
        ]

        for atributo, valor in posibles_fechas:
            meta_fecha = soup.find(
                "meta",
                attrs={atributo: valor}
            )

            if meta_fecha and meta_fecha.get("content"):
                fecha = meta_fecha["content"].strip()
                break

        # ---------------------------
        # SECCIÓN ORIGINAL
        # ---------------------------
        seccion = ""

        meta_seccion = soup.find(
            "meta",
            attrs={"property": "article:section"}
        )

        if meta_seccion and meta_seccion.get("content"):
            seccion = meta_seccion["content"].strip()

        # Fallback si el sitio no informa la sección
        if not seccion:
            mapa_secciones = {
                "politica": "Política",
                "economia": "Economía",
                "deportes": "DT",
                "ciencia_tecnologia": "Tecnología y Ciencias"
            }

            seccion = mapa_secciones.get(etiqueta, "")

        return {
            "bajada": bajada,
            "fecha": fecha,
            "seccion_original": seccion
        }

    except Exception as e:
        return {
            "bajada": "",
            "fecha": "",
            "seccion_original": ""
        }

In [80]:
resultados_meta = []

total = len(df_muestra)

for i, fila in df_muestra.iterrows():

    meta = obtener_meta_articulo(
        fila["url"],
        fila["etiqueta"]
    )

    resultados_meta.append(meta)

    if (i + 1) % 20 == 0:
        print(
            f"Procesadas {i + 1} de {total} noticias"
        )

    time.sleep(1)

print("Proceso terminado.")

Procesadas 20 de 360 noticias
Procesadas 40 de 360 noticias
Procesadas 60 de 360 noticias
Procesadas 80 de 360 noticias
Procesadas 100 de 360 noticias
Procesadas 120 de 360 noticias
Procesadas 140 de 360 noticias
Procesadas 160 de 360 noticias
Procesadas 180 de 360 noticias
Procesadas 200 de 360 noticias
Procesadas 220 de 360 noticias
Procesadas 240 de 360 noticias
Procesadas 260 de 360 noticias
Procesadas 280 de 360 noticias
Procesadas 300 de 360 noticias
Procesadas 320 de 360 noticias
Procesadas 340 de 360 noticias
Procesadas 360 de 360 noticias
Proceso terminado.


In [81]:
df_meta = pd.DataFrame(resultados_meta)

df_final = df_muestra.copy()

df_final["bajada"] = df_meta["bajada"]
df_final["fecha"] = df_meta["fecha"]
df_final["seccion_original"] = df_meta["seccion_original"]
df_final["medio"] = "El Comercio"

In [82]:
df_final = df_final[
    [
        "titular",
        "bajada",
        "medio",
        "seccion_original",
        "url",
        "fecha",
        "etiqueta"
    ]
]

print("Dimensión final:", df_final.shape)

df_final.head()

Dimensión final: (360, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Presidente del TC sobre leyes aprobadas por el...,El presidente del Tribunal Constitucional (TC)...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/presid...,2026-09-07T20:51:41-05:00,politica
1,Caso Árbitros-Odebrecht: audiencia de control ...,La Corte Superior Nacional de Justicia Penal E...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/caso-a...,2026-10-12T12:48:23-05:00,politica
2,Flavio Figallo del Partido del Buen Gobierno: ...,"El senador del Partido del Buen Gobierno, Flav...",El Comercio,Congreso,https://elcomercio.pe/politica/congreso/flavio...,2026-10-16T11:47:20-05:00,politica
3,Unión Europea y Minem acuerdan creación de mes...,La Unión Europea (UE) y el Ministerio de Energ...,El Comercio,Gobierno,https://elcomercio.pe/politica/gobierno/union-...,2026-10-13T19:19:04-05:00,politica
4,Fiscalía realizó diligencias en Ministerio de ...,Las diligencias estuvieron a cargo de la fisca...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/albert...,2026-08-27T13:43:00-05:00,politica


In [83]:
print("Valores vacíos por columna:")
print()

for columna in df_final.columns:
    vacios = (
        df_final[columna]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(columna, "->", vacios)
    

Valores vacíos por columna:

titular -> 0
bajada -> 0
medio -> 0
seccion_original -> 0
url -> 0
fecha -> 0
etiqueta -> 0


In [84]:
ruta_csv = DATOS / "noticias.csv"

df_final.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado en:", ruta_csv)
print("Dimensión:", df_final.shape)

Archivo guardado en: datos\noticias.csv
Dimensión: (360, 7)


In [85]:
df_verificacion = pd.read_csv(ruta_csv)

print("Dimensión leída:", df_verificacion.shape)
df_verificacion.head()

Dimensión leída: (360, 7)


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Presidente del TC sobre leyes aprobadas por el...,El presidente del Tribunal Constitucional (TC)...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/presid...,2026-09-07T20:51:41-05:00,politica
1,Caso Árbitros-Odebrecht: audiencia de control ...,La Corte Superior Nacional de Justicia Penal E...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/caso-a...,2026-10-12T12:48:23-05:00,politica
2,Flavio Figallo del Partido del Buen Gobierno: ...,"El senador del Partido del Buen Gobierno, Flav...",El Comercio,Congreso,https://elcomercio.pe/politica/congreso/flavio...,2026-10-16T11:47:20-05:00,politica
3,Unión Europea y Minem acuerdan creación de mes...,La Unión Europea (UE) y el Ministerio de Energ...,El Comercio,Gobierno,https://elcomercio.pe/politica/gobierno/union-...,2026-10-13T19:19:04-05:00,politica
4,Fiscalía realizó diligencias en Ministerio de ...,Las diligencias estuvieron a cargo de la fisca...,El Comercio,Justicia,https://elcomercio.pe/politica/justicia/albert...,2026-08-27T13:43:00-05:00,politica


In [87]:
%pip install plotly

  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
Using cached plotly-7.1.0-py3-none-any.whl (9.7 MB)
Using cached narwhals-2.27.1-py3-none-any.whl (483 kB)

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhal


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [88]:
import plotly.express as px

In [89]:
import plotly.express as px

df_final["longitud_titular"] = df_final["titular"].astype(str).str.len()

df_final["fecha"] = pd.to_datetime(
    df_final["fecha"],
    errors="coerce",
    utc=True
)

df_final["fecha_dia"] = df_final["fecha"].dt.date

In [92]:
%pip install nbformat

  Using cached nbformat-5.11.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached fastjsonschema-2.22.2-py3-none-any.whl.metadata (2.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-2026.9.1-cp313-cp313-win_amd64.whl.metadata (4.2 kB)
Using cached nbformat-5.11.1-py3-none-any.whl (79 kB)
Using cached fastjsonschema-2.22.2-py3-none-any.whl (27 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using cached rpds_py-2026.9.1-cp313-cp313-win_amd64.whl (222 kB)

   ------ --------------------------------- 1/6 [fastjsonschema]
   ------------- -------------------------- 2/6 [referencing]
   -------------------- ------------------- 3/6 [jsonschema-specificati


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [93]:
import plotly.express as px

In [95]:
conteo_categoria = (
    df_final["etiqueta"]
    .value_counts()
    .reset_index()
)

conteo_categoria.columns = ["categoria", "cantidad"]

fig1 = px.bar(
    conteo_categoria,
    x="categoria",
    y="cantidad",
    title="La muestra mantiene una distribución relativamente equilibrada entre categorías"
)

fig1.show(renderer="browser")

In [96]:
conteo_fecha = (
    df_final
    .groupby("fecha_dia")
    .size()
    .reset_index(name="cantidad")
)

fig2 = px.line(
    conteo_fecha,
    x="fecha_dia",
    y="cantidad",
    markers=True,
    title="La publicación de noticias se concentra en determinados días del periodo analizado"
)

fig2.show(renderer="browser")

In [97]:
fig3 = px.histogram(
    df_final,
    x="longitud_titular",
    nbins=30,
    title="La mayoría de los titulares presenta una longitud intermedia"
)

fig3.show(renderer="browser")

In [98]:
titulares_cortos = (
    df_final[
        ["titular", "longitud_titular"]
    ]
    .sort_values("longitud_titular")
    .head(5)
)

titulares_largos = (
    df_final[
        ["titular", "longitud_titular"]
    ]
    .sort_values("longitud_titular", ascending=False)
    .head(5)
)

print("5 titulares más cortos:")
display(titulares_cortos)

print("5 titulares más largos:")
display(titulares_largos)

5 titulares más cortos:


,titular,longitud_titular
28,BTS,3
33,ONPE,4
82,Nobel,5
90,Panamá,6
56,Fujimori,8


5 titulares más largos:


,titular,longitud_titular
345,“Con Mano Menezes el mejor jugador de la selec...,171
306,“Creo que puedo llegar a un top 3 sin que nadi...,167
22,Tomás Gálvez: “El Congreso tiene que estar a l...,164
297,“El día de mañana me encantaría ver a tantas n...,160
29,Juan Sheput sobre eventual suspensión del viaj...,160


## Conclusiones de la recolección y limpieza

Se construyó una base final de 360 noticias reales de El Comercio, distribuidas entre las categorías política, economía, deportes y ciencia_tecnologia.

La información fue obtenida mediante los RSS oficiales y las páginas de secciones y subsecciones del medio. Posteriormente se eliminaron registros duplicados utilizando la URL como identificador principal y se verificó que no existieran titulares ni campos obligatorios vacíos.

La muestra final contiene las siete variables requeridas: titular, bajada, medio, seccion_original, url, fecha y etiqueta.

El análisis exploratorio permitió observar la distribución de noticias por categoría y fecha, así como las diferencias en la longitud de los titulares. La base obtenida será utilizada posteriormente para la etapa de clasificación automática de noticias.

In [99]:
print("=== VALIDACIÓN FINAL ===")
print("Dimensión:", df_final.shape)
print("URLs duplicadas:", df_final["url"].duplicated().sum())
print("Titulares duplicados:", df_final["titular"].duplicated().sum())
print("Valores nulos:", df_final.isna().sum().sum())

print("\nCategorías:")
print(df_final["etiqueta"].value_counts())

print("\nColumnas:")
print(df_final.columns.tolist())

=== VALIDACIÓN FINAL ===
Dimensión: (360, 9)
URLs duplicadas: 0
Titulares duplicados: 0
Valores nulos: 0

Categorías:
etiqueta
politica              95
economia              95
ciencia_tecnologia    95
deportes              75
Name: count, dtype: int64

Columnas:
['titular', 'bajada', 'medio', 'seccion_original', 'url', 'fecha', 'etiqueta', 'longitud_titular', 'fecha_dia']
